# Measurement and analysis handling walkthrough

This notebook demonstrates the optional in-memory handling layer. A file-agnostic pandas adapter creates shared-grid signals, while `Signal` and `Measurement` also preserve independent coordinate grids and units. `Analysis` composes the standalone numerical functions and records provenance. No file loading, persistence, or plotting behavior is implied.

In [ ]:
import numpy as np
import pandas as pd

from signal_analyzer.analysis import Analysis
from signal_analyzer.measurement import Measurement, Signal
from signal_analyzer.synthetic_signals import simulate_bearing_fault
from signal_analyzer.tabular import ChannelSpec, measurement_from_dataframe

## 1. Build signals without forcing a common time base

The vibration and speed channels are synchronous at 4096 samples/s. A slower process channel has its own rate and start time. The measurement stores these grids independently.

In [ ]:
fs = 4096.0
fault_order = 4.8
t, speed, _, x = simulate_bearing_fault(
    fs=fs,
    duration=4.0,
    speed_min=2.5,
    speed_max=10.0,
    fault_order=fault_order,
    resonance_frequency=900.0,
    noise_std=0.03,
    seed=21,
)

frame = pd.DataFrame({"vibration": x, "speed": speed})
measurement = measurement_from_dataframe(
    frame,
    sample_rate=fs,
    coordinate_unit="s",
    channels={
        "vibration": ChannelSpec("vibration", value_unit="g"),
        "speed": ChannelSpec("speed", value_unit="rev/s"),
    },
    metadata={"source": "deterministic synthetic DataFrame"},
)

process_fs = 20.0
process_values = np.linspace(0.2, 0.8, int(4 * process_fs))
measurement.add_signal(
    "process",
    Signal(
        process_values, sample_rate=process_fs, start=0.1,
        value_unit="normalized", coordinate_unit="s",
    ),
)

In [ ]:
for name, signal in measurement.signals.items():
    print(
        f"{name:10s}: {len(signal.values):5d} samples, "
        f"rate={signal.inferred_sample_rate():7.1f}, "
        f"start={signal.coordinates[0]:.3f} {signal.coordinate_unit}"
    )

## 2. Configure analysis and add derived results

The facade holds defaults but calls the standalone functions. Derived arrays are added as named `Signal` objects. Scalar indicators return unit-aware `AnalysisResult` objects and are retained when `output_name` is supplied. Alignment is explicit: the process channel is interpolated only after the workflow requests a shared overlap grid.

In [ ]:
analysis = Analysis(measurement, resolution=64, oversample=4)

vibration_for_process, process_aligned = analysis.align_pair(
    "vibration", "process",
    "vibration_process_window", "process_aligned",
)

vibration_down = analysis.downsample(
    "vibration", "vibration_down", down=2
)
vibration_psd = analysis.psd(
    "vibration_down", "vibration_psd",
    block_size=1024, overlap=0.5, window="hann",
)
time_stats = analysis.signal_statistics(
    "vibration", lower=0.5, upper=1.5,
    output_name="vibration_time_statistics",
)
resonance_stats = analysis.psd_band_statistics(
    "vibration_psd", lower=700, upper=1000,
    output_name="resonance_band_statistics",
)

print(f"downsampled rate: {vibration_down.sample_rate:g} samples/s")
print(f"explicitly aligned process samples: {len(process_aligned.values)}")
print(f"selected time-zone RMS: {time_stats.values['rms']:.3f} {time_stats.units['rms']}")
print(f"resonance-band RMS: {resonance_stats.values['band_rms']:.3f} {resonance_stats.units['band_rms']}")
print(vibration_psd.metadata)

## 3. Run the envelope-order happy path

The carrier band is temporal, so it is demodulated before trimming and position resampling. The happy path retains envelope, trimmed, spatial, and order-spectrum intermediates.

In [ ]:
order_spectrum = analysis.envelope_order_spectrum(
    "vibration",
    "speed",
    output_prefix="bearing",
    band=(700, 1100),
    lowpass=100,
    block_size=512,
    trim_samples=int(0.25 * fs),
)
fault_stats = analysis.amplitude_band_statistics(
    "bearing_order_spectrum", lower=4, upper=6,
    output_name="fault_order_statistics",
)

print(f"configured fault order: {fault_order:.2f}")
print(f"measured band peak: {fault_stats.values['peak_coordinate']:.2f}")
print(f"practical order limit: {analysis.resolution / 2.56:.1f}")
print(order_spectrum.metadata)

## 4. Inspect provenance and retained intermediates

Provenance records are plain operation descriptions. They are intentionally storage-neutral; persistence will be designed separately after this schema has been exercised.

In [ ]:
print("signals:")
for name in measurement.signals:
    print(f"  {name}")

print("\nresults:")
for name, result in measurement.results.items():
    print(f"  {name}: {result.kind}")

print("\nprovenance:")
for record in measurement.provenance:
    print(
        f"  {record.output:28s} <- {record.operation:22s} "
        f"inputs={record.inputs}"
    )

## Takeaways

- Signals retain independent coordinate grids until `align_pair` explicitly selects an overlap and reference grid.
- Numerical work stays in standalone functions; `Analysis` composes them and records unit-aware signals, scalar results, and provenance.
- Built-in amplitude and PSD spectra carry explicit kind and normalization metadata.
- Scalar indicators are currently returned directly rather than stored as measurement artifacts.
- Persistence and plotting remain separate future layers.